# A network step inside a single-cell workflow

You start with an `AnnData`. You end with a column in `adata.obs`. In between, two
**prior knowledge networks** carry the measurements somewhere a gene list cannot go.

Everything here is real and downloaded by this notebook:

- **Kang et al. 2018** — PBMCs from eight donors, half stimulated with interferon-beta.
- **CollecTRI** — which transcription factor regulates which gene, and how.
- **OmniPath** — which protein acts on which protein, upstream of transcription.

Two resources answering two different questions, on **one** `AnnNet`, as two slices.
Then `decoupler` scores regulator activity from the first, and CORNETO's **CARNIVAL**
fits a signalling sub-network per sample from the second.

## 1. Their object, unchanged

In [1]:
import warnings, time
import numpy as np
import pertpy as pt
import scanpy as sc
import decoupler as dc

warnings.simplefilter('ignore')

adata = pt.dt.kang_2018()
adata

/home/l1boll/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnnData object with n_obs × n_vars = 24673 × 15706
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name'
    obsm: 'X_pca', 'X_umap'

24,673 cells by 15,706 genes, `label` in `{ctrl, stim}`, eight cell types. Nothing here
is ours.

The methods below work on samples rather than cells, so pseudobulk first — one profile
per cell type per stimulation. Sixteen samples. This is `decoupler` and `scanpy` doing
what they always do; the network step has not started.

In [2]:
pdata = dc.pp.pseudobulk(adata, sample_col='label', groups_col='cell_type', mode='sum')
dc.pp.filter_by_expr(pdata, min_count=5, min_total_count=10)
sc.pp.normalize_total(pdata, target_sum=1e6)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)

conditions = [str(name) for name in pdata.obs_names]
measured = {str(name) for name in pdata.var_names}
print(pdata.shape)
conditions[:4]

(16, 3073)


['ctrl_B cells',
 'stim_B cells',
 'ctrl_CD14+ Monocytes',
 'stim_CD14+ Monocytes']

## 2. Two networks, one object

**CollecTRI** is a gene regulatory network. `decoupler` scores transcription-factor
activity from it. We keep the interactions whose *target* was measured — a regulator is
scored **from** its targets, so it need not be measured itself.

Two things about the call that builds the graph:

- **Conditions become an aspect.** The sixteen samples are layers, so a value can be
  keyed by *(gene, sample)* rather than living in a dict beside the object.
- **Edge ids are derived from content**, not row position. Re-sort the table and every
  id is identical. `slice='regulon'` is what answers *which edges came from this table*,
  so no list of ids is kept on the side.

In [3]:
import annnet as an
from annnet import exp

collectri = dc.op.collectri(organism='human')
net = collectri[collectri['target'].isin(measured)]
net = (net.rename(columns={'weight': 'effect'})[['source', 'target', 'effect']]
          .drop_duplicates(subset=['source', 'target']))

G = an.from_edge_frame(
    net, directed=True, sign='effect', slice='regulon',
    aspects={'condition': conditions},
)
G.provenance.record('CollecTRI', version='decoupler.op.collectri', reader='from_edge_frame')
print(f'{len(list(G.edges()))} regulatory interactions')

7335 regulatory interactions


Now the signalling network, into a **second slice on the same object**. Signal cannot
travel through a protein the assay did not see, so it is restricted to what was
measured.

`on_conflict='rename'` matters here: the two resources overlap, and a derived id that is
already taken has to be resolved deliberately rather than silently dropped or
overwritten.

In [4]:
import omnipath as op

raw = op.interactions.OmniPath.get(genesymbols=True, organism='human')
signalling = raw[
    raw['consensus_direction']
    & (raw['consensus_stimulation'] ^ raw['consensus_inhibition'])
].copy()
signalling['effect'] = np.where(signalling['consensus_stimulation'], 1, -1)
signalling = (
    signalling[['source_genesymbol', 'target_genesymbol', 'effect']]
    .rename(columns={'source_genesymbol': 'source', 'target_genesymbol': 'target'})
    .drop_duplicates(subset=['source', 'target'])
)
signalling = signalling[
    signalling['source'].isin(measured) & signalling['target'].isin(measured)
]

an.add_edges_from_frame(
    G, signalling, directed=True, sign='effect', slice='signalling', on_conflict='rename',
)
G.provenance.record('OmniPath', version='omnipath.interactions.OmniPath',
                    reader='add_edges_from_frame')
G.attrs.set_node_attrs_bulk({n: {'symbol': n, 'kind': 'protein'} for n in G.nodes()})
print(G.slices.list(), '|', len(list(G.edges())), 'edges total')
G

['default', 'regulon', 'signalling'] | 8421 edges total


AnnNet object with n_nodes × n_edges = 2111 × 8421
    directed: True
    slices: ['default', 'regulon', 'signalling']
    aspects: ['condition']
    supra_nodes (node × layer rows): 2111
    obs: ['symbol', 'kind']
    var: ['sign']
    uns: ['__provenance__']

`G.provenance()` is each resource reporting its own version and retrieval date. Most networks are pasted into an analysis as a bare table and lose that on the way.

In [5]:
G.provenance()

name,format,uri,version,retrieved,checksum,reader,rows
str,str,null,str,str,null,str,i64
"""edge frame""","""table""",null,null,"""2026-09-05T03:53:58.120084+00:…",null,"""from_edge_frame""",7335
"""CollecTRI""",null,null,"""decoupler.op.collectri""","""2026-09-05T03:53:58.120237+00:…",null,"""from_edge_frame""",null
"""OmniPath""",null,null,"""omnipath.interactions.OmniPath""","""2026-09-05T03:54:04.053404+00:…",null,"""add_edges_from_frame""",null


## 3. Reading the graph

The edge table reports **node ids**, and the layer each endpoint sits in is its own
column — which is what makes it joinable against anything else. `in_slice=` keeps only
one slice's rows; `slice=` would join that slice's attributes onto *every* row instead.

In [6]:
G.views.edges(in_slice='signalling', include_hyper=False).select(
    ['edge_id', 'source', 'target', 'sign']
).head(5)

edge_id,source,target,sign
str,str,str,f64
"""e:97d795eecd2bfa10""","""CASP3""","""EIF3J""",-1.0
"""e:6bfccc0ba478eb9e""","""MAPKAPK2""","""HNRNPA0""",1.0
"""e:9a3eec6454795d49""","""MAPKAPK2""","""ZFP36""",-1.0
"""e:942b386c3283d70e""","""CREB1""","""MIF""",1.0
"""e:1efe4eb85aba9804""","""CASP3""","""ACIN1""",1.0


## 4. What a method needs, checked before it runs

A solver handed the wrong shape of graph does not fail. It fits something, and a
plausible number is worse than a crash because it gets published.

Each adapter declares its requirement as a value you can read without running anything.
`check` reports contract problems and shape problems together.

In [7]:
print(exp.sysbio.methods.decoupler.SPEC)
print(exp.sysbio.methods.corneto.SPEC)
print()
print('contract :', exp.vocabulary.check(G) or 'clean')
print('decoupler:', exp.vocabulary.check(G, method=exp.sysbio.methods.decoupler.SPEC) or 'as needed')
print('carnival :', exp.vocabulary.check(G, method=exp.sysbio.methods.corneto.SPEC) or 'as needed')

decoupler needs: sign on every edge, directed, dyadic
corneto.carnival needs: sign on every edge, directed, dyadic

contract : clean


decoupler: as needed
carnival : as needed


The refusal is worth more than the pass. Here is the same check on a network whose edges
carry no direction of effect — which a scoring method would read perfectly happily,
treating activation and repression alike.

In [8]:
unsigned = an.from_edge_frame([{'source': 'TF1', 'target': 'GENE1'}])
try:
    exp.vocabulary.check(unsigned, method=exp.sysbio.methods.decoupler.SPEC, strict=True)
except exp.vocabulary.ContractViolation as refusal:
    print(refusal)

'decoupler' needs all 1 edges to carry 'sign'; 1 do not (e.g. ['e:dff78d49fa5f990e'])


Note what the decoupler spec does **not** say: `bipartite`. A regulon looks bipartite,
and declaring it would be free rigour — except that **921 of CollecTRI's 1,185
regulators are themselves targets**. A spec that declared it would refuse the resource
the method exists to score, and what a user learns from that is to skip the check.

In [9]:
sources, targets = set(collectri['source']), set(collectri['target'])
print(f'regulators that are themselves targets: {len(sources & targets)} of {len(sources)}')
print('bipartite declared by the spec:', exp.sysbio.methods.decoupler.SPEC.bipartite)

regulators that are themselves targets: 921 of 1185
bipartite declared by the spec: False


## 5. Attach — a join, with the multiplicity stated

`attach` is a join between two vocabularies that do not correspond one to one. What did
not match comes back **as data**, before anyone has to ask.

Two things this does *not* do, and both matter:

- **It does not copy the matrix.** The graph holds two index maps onto the array the
  `AnnData` already owns. The measurements stay where they were.
- **It does not place a node-layer per cell.** Reading an attached array never consults
  presence, so `place='matched'` creates node-layers only for the genes the join reached
  — the size of the network, not the size of the assay.

In [10]:
t0 = time.perf_counter()
report = exp.sysbio.attach(
    G, pdata, aspect='condition', on='symbol',
    values={'expression': None}, multiplicity='allow',
)
print(f'{time.perf_counter() - t0:.2f} s')
report

0.02 s


AttachReport(layers=16, matched=1462, unmapped=1611, unmeasured=649, coverage=47.6%)

Say the coverage out loud. A network covers a fraction of the transcriptome — the point is that the fraction is *reported* rather than silently dropped.

In [11]:
print(f'coverage   : {report.coverage:.1%}')
print(f'unmapped   : {len(report.unmapped_ids)} measured genes reached no node')
print(f'unmeasured : {len(report.unmeasured_ids)} nodes nothing measured')
report.mapping.head(5)

coverage   : 47.6%
unmapped   : 1611 measured genes reached no node
unmeasured : 649 nodes nothing measured


input,resolved,status
str,str,str
"""ISG15""","""ISG15""","""ok"""
"""SDF4""",null,"""unmapped"""
"""UBE2J2""",null,"""unmapped"""
"""CPSF3L""",null,"""unmapped"""
"""AURKAIP1""","""AURKAIP1""","""ok"""


## 6. Reading it back as an array

This is what a method is handed. A frame of Python objects has to be unpacked before any
arithmetic; this is the arithmetic's own shape, plus the two labels that put an answer
back on the right rows.

In [12]:
t0 = time.perf_counter()
block = G.layers.matrix('expression')
print(f'{block} in {(time.perf_counter() - t0) * 1000:.0f} ms')
print('connected:', exp.sysbio.connected(G), exp.sysbio.measurements(G))

ValueMatrix('expression', 16 layers x 1462 nodes) in 1 ms
connected: True ['expression']


## 7. Transcription-factor activity, from the regulon slice

The arithmetic is `decoupler`'s, unchanged — the test suite pins it to floating point
against calling `dc.mt.ulm` on the equivalent DataFrame. What the adapter contributes is
the declaration before, reading the regulon off the graph, and writing the answer back
**additively**.

`slice='regulon'` is why the two networks can live on one object: the signalling edges
are right there and are not part of this.

In [13]:
t0 = time.perf_counter()
activity = exp.sysbio.methods.decoupler.run(
    G, pdata, aspect='condition', slice='regulon', into_slice='scored', tmin=5,
)
print(f'{time.perf_counter() - t0:.1f} s')
print(activity)
activity.scores.iloc[:4, :6]

0.3 s
DecouplerResult(method='ulm', sources=315, dropped=455, slice='scored', key='score')


,AHR,AIRE,AP1,APEX1,AR,ARNT
ctrl_B cells,-0.153312,-2.146860,-2.843462,-1.112572,-1.459129,0.113232
stim_B cells,0.124541,-2.943634,-2.911075,-0.047172,-1.712484,-0.197568
ctrl_CD14+ Monocytes,0.165394,4.042446,2.829850,3.019722,1.890057,-0.192403
stim_CD14+ Monocytes,0.097044,2.960792,3.480973,2.344534,2.155722,0.810997


`activity.dropped` is the regulators `tmin` left out. One missing because its targets
were not measured looks exactly like one that scored zero, so it comes back **named**
rather than passed over.

`activity.placed` is the node-layers created so the scores had somewhere to sit — a
regulator whose own gene the assay never measured has none otherwise. Identity and
values are separate questions, and this is where that shows.

In [14]:
print(f'{len(activity.sources)} scored, {len(activity.dropped)} dropped for want of targets')
print(f'{activity.placed} node-layers placed for the scores')
activity.dropped[:8]

315 scored, 455 dropped for want of targets
3984 node-layers placed for the scores


['ABL1', 'ADNP', 'ADNP2', 'AHRR', 'AIP', 'APBB1', 'ARHGAP35', 'ARID1A']

## 8. Back into `adata.obs`

The scores are per regulator per sample and went onto the graph as node-layer
attributes. The number that belongs to the **network** rather than to any one regulator
— here, how many regulators are active in each sample — is a per-condition scalar, and
that is what `obs` is for.

**Write-back is a projection, not a round trip.** `obs`, `var` and `layers` receive
numbers; the topology stays on the graph.

In [15]:
exp.sysbio.write_back(G, pdata, key='n_active', aspect='condition', into='obs')
pdata.obs[['label', 'cell_type', 'n_active']].head(8)

,label,cell_type,n_active
ctrl_B cells,ctrl,B cells,88
stim_B cells,stim,B cells,77
ctrl_CD14+ Monocytes,ctrl,CD14+ Monocytes,73
stim_CD14+ Monocytes,stim,CD14+ Monocytes,91
ctrl_CD4 T cells,ctrl,CD4 T cells,86
stim_CD4 T cells,stim,CD4 T cells,71
ctrl_CD8 T cells,ctrl,CD8 T cells,100
stim_CD8 T cells,stim,CD8 T cells,76


That is the loop closed: their object, with the result in it.

Everything from here is what the object can do that the column cannot.

## 9. Which interactions carried the signal, in which cell type

Interferon-beta signals through the type I interferon receptor. Mark it perturbed in the
stimulated samples — a per-condition fact, so it is stored per condition.

In [16]:
receptors = [n for n in ('IFNAR1', 'IFNAR2') if n in set(G.nodes())]
stimulated = [c for c in conditions if c.startswith('stim')]

G.layers.place(receptors, [(c,) for c in stimulated])
G.layers.set_node_attrs_bulk(
    {(r, (c,)): 1.0 for r in receptors for c in stimulated}, key='perturbation',
)
print(receptors, '|', len(stimulated), 'stimulated samples')

['IFNAR2'] | 8 stimulated samples


CARNIVAL now fits, per condition, the sub-network of the signalling slice that best
explains those regulator activities from that perturbation.

`outputs=activity.key` is the join between the two steps: it names the attribute the
previous cell wrote. **No intermediate dictionary**, no realignment, and no way for the
two steps to disagree about which condition is which.

`top=` is the cost knob — CARNIVAL's work grows with the number of measurements it has
to explain.

In [17]:
pair = stimulated[:2]
t0 = time.perf_counter()
fit = exp.sysbio.methods.corneto.run(
    G,
    inputs='perturbation',
    outputs=activity.key,
    aspect='condition',
    conditions=pair,
    slice='signalling',
    top=25,
    lambda_reg=0.1,
)
print(f'{time.perf_counter() - t0:.1f} s')
fit

Unreachable vertices for sample: 0


Unreachable vertices for sample: 0


2.9 s


CarnivalResult(stim_B cells=18, stim_CD14+ Monocytes=15, key='activity')

Here is the result a `source`/`target`/`weight` DataFrame cannot express: one value per **edge, per condition**, signed. One call.

In [18]:
G.slices.edge_frame(slices=fit.slices, attrs=['activity'], format='long').head(8)

edge_id,slice_id,attr,value
str,str,str,f64
"""e:0b0259fb4c897f3c""","""carnival__stim_B cells""","""activity""",-1.0
"""e:0b33d62fcfa76c4a""","""carnival__stim_B cells""","""activity""",-1.0
"""e:0e90a412c693b2a8""","""carnival__stim_B cells""","""activity""",-1.0
"""e:1120dc821bf01d6f""","""carnival__stim_B cells""","""activity""",-1.0
"""e:19dbbcb1c00544cd""","""carnival__stim_B cells""","""activity""",-1.0
"""e:1e08942ea7d5ff35""","""carnival__stim_B cells""","""activity""",1.0
"""e:235ebc4b298e4e9a""","""carnival__stim_B cells""","""activity""",null
"""e:2655759520e085ce""","""carnival__stim_B cells""","""activity""",null


And the comparison, which is the thing you actually wanted: which interactions carry interferon signal in one cell type and not the other.

In [19]:
comparison = G.slices.compare(*fit.slices, axis='edges')
import collections
print(collections.Counter(row['status'] for row in comparison.to_dicts()))
comparison.head(8)

Counter({'a_only': 14, 'b_only': 11, 'both': 4})


edge_id,status
str,str
"""e:0b0259fb4c897f3c""","""a_only"""
"""e:0b33d62fcfa76c4a""","""both"""
"""e:0e90a412c693b2a8""","""both"""
"""e:1120dc821bf01d6f""","""a_only"""
"""e:19dbbcb1c00544cd""","""both"""
"""e:1e08942ea7d5ff35""","""a_only"""
"""e:235ebc4b298e4e9a""","""b_only"""
"""e:2655759520e085ce""","""b_only"""


Joined to the endpoints, that is a readable answer to a question a gene list cannot be asked.

In [20]:
import polars as pl

edges = G.views.edges(in_slice='signalling').select(['edge_id', 'source', 'target', 'sign'])
(
    comparison.join(edges, on='edge_id')
    .filter(pl.col('status') != 'both')
    .select(
        pl.col('source'),
        pl.when(pl.col('sign') == 1).then(pl.lit('activates'))
          .otherwise(pl.lit('inhibits')).alias('effect'),
        pl.col('target'),
        pl.when(pl.col('status') == 'a_only').then(pl.lit(fit.conditions[0]))
          .otherwise(pl.lit(fit.conditions[1])).alias('carries signal in'),
    )
    .sort(['carries signal in', 'source'])
    .head(10)
)

source,effect,target,carries signal in
str,str,str,str
"""ATM""","""activates""","""PPP2R5C""","""stim_B cells"""
"""JAK1""","""activates""","""STAT6""","""stim_B cells"""
"""MAPK1""","""activates""","""CEBPB""","""stim_B cells"""
"""PIM1""","""activates""","""HIF1A""","""stim_B cells"""
"""PIM1""","""activates""","""RUNX3""","""stim_B cells"""
"""PPP2CA""","""activates""","""MYC""","""stim_B cells"""
"""PPP2CA""","""inhibits""","""ATM""","""stim_B cells"""
"""PPP2R2A""","""inhibits""","""PPP2CA""","""stim_B cells"""
"""PPP2R5C""","""activates""","""ATF1""","""stim_B cells"""


## 10. Two priors and three fits, on one object

Nothing overwrote anything. The two source networks, the regulator scoring and the
per-condition CARNIVAL fits are all on the same object, and any pair of them can be
diffed.

In [21]:
print(G.slices.list(include_default=True))
G.slices.compare('signalling', fit.slices[0], axis='edges').head(5)

['default', 'regulon', 'signalling', 'scored', 'carnival__stim_B cells', 'carnival__stim_CD14+ Monocytes']


edge_id,status
str,str
"""e:003fc3a0d4a88a7e""","""a_only"""
"""e:00543ced39ef03a4""","""a_only"""
"""e:005f50770fbe132f""","""a_only"""
"""e:00da7878a6b001d0""","""a_only"""
"""e:011abca1ebb22d10""","""a_only"""


---

## What was ours

```python
G = an.from_edge_frame(net, sign='effect', slice='regulon', aspects={'condition': ...})
an.add_edges_from_frame(G, signalling, sign='effect', slice='signalling', on_conflict='rename')
G.provenance.record(...)
exp.vocabulary.check(G, method=exp.sysbio.methods.decoupler.SPEC)
report = exp.sysbio.attach(G, pdata, aspect='condition', on='symbol')
block = G.layers.matrix('expression')
activity = exp.sysbio.methods.decoupler.run(G, pdata, aspect='condition', slice='regulon')
exp.sysbio.write_back(G, pdata, key='n_active', aspect='condition', into='obs')
G.layers.place(receptors, [...]); G.layers.set_node_attrs_bulk({...}, key='perturbation')
fit = exp.sysbio.methods.corneto.run(G, inputs='perturbation', outputs=activity.key, ...)
G.slices.edge_frame(slices=fit.slices, attrs=['activity'])
G.slices.compare(*fit.slices, axis='edges')
```

Twelve calls. No helper functions, no index bookkeeping, no dictionary keyed on a tuple
somebody has to remember the order of — and **no arithmetic of ours**: the scores are
decoupler's and the fits are CORNETO's, both pinned to floating point by the test suite.